In [18]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# ORIGINAL FILES
# --------------------------------------------------
priceFiles = [
    "prices_round_1_day_0.csv",
    "prices_round_1_day_1.csv",
    "prices_round_1_day_2.csv",
]

tradeFiles = [
    "trades_round_1_day_0.csv",
    "trades_round_1_day_1.csv",
    "trades_round_1_day_2.csv",
]

# explicit day mapping for ALL files
fileDayMap = {
    "prices_round_1_day_0.csv": 0,
    "prices_round_1_day_1.csv": 1,
    "prices_round_1_day_2.csv": 2,
    "trades_round_1_day_0.csv": 0,
    "trades_round_1_day_1.csv": 1,
    "trades_round_1_day_2.csv": 2,
}

outputDir = Path("cleaned_round1")
outputDir.mkdir(exist_ok=True)

priceNumericCols = [
    "day", "timestamp",
    "bid_price_1", "bid_volume_1",
    "bid_price_2", "bid_volume_2",
    "bid_price_3", "bid_volume_3",
    "ask_price_1", "ask_volume_1",
    "ask_price_2", "ask_volume_2",
    "ask_price_3", "ask_volume_3",
    "mid_price", "profit_and_loss"
]

tradeNumericCols = [
    "day", "timestamp", "price", "quantity"
]

def loadFileWithForcedDay(path, numericCols):
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip()

    fileName = Path(path).name
    if fileName not in fileDayMap:
        raise ValueError(f"Filename not found in fileDayMap: {fileName}")

    # FORCE day from filename for all files
    df["day"] = fileDayMap[fileName]

    for col in numericCols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def addBookFeatures(df):
    out = df.copy()
    out["globalTs"] = out["day"] * 100000 + out["timestamp"]

    if {"bid_price_1", "ask_price_1"}.issubset(out.columns):
        out["spread"] = out["ask_price_1"] - out["bid_price_1"]

    if {"bid_price_1", "ask_price_1", "bid_volume_1", "ask_volume_1"}.issubset(out.columns):
        bidVol = out["bid_volume_1"].abs()
        askVol = out["ask_volume_1"].abs()
        totalVol = bidVol + askVol

        out["microprice"] = (
            out["bid_price_1"] * askVol + out["ask_price_1"] * bidVol
        ) / totalVol.replace(0, pd.NA)

        out["imbalance"] = (
            (bidVol - askVol) / totalVol.replace(0, pd.NA)
        )

    return out

def makeTradeAgg(tradesDf):
    agg = (
        tradesDf.groupby(["day", "timestamp"], as_index=False)
        .agg(
            tradeCount=("price", "size"),
            totalQty=("quantity", "sum"),
            avgTradePrice=("price", "mean"),
            maxTradePrice=("price", "max"),
            minTradePrice=("price", "min")
        )
    )
    agg["globalTs"] = agg["day"] * 100000 + agg["timestamp"]
    return agg

# --------------------------------------------------
# LOAD ORIGINAL FILES WITH FORCED DAY
# --------------------------------------------------
allPrices = pd.concat(
    [loadFileWithForcedDay(f, priceNumericCols) for f in priceFiles],
    ignore_index=True
)

allTrades = pd.concat(
    [loadFileWithForcedDay(f, tradeNumericCols) for f in tradeFiles],
    ignore_index=True
)

# --------------------------------------------------
# SPLIT BY PRODUCT
# --------------------------------------------------
ashPricesDf = allPrices[allPrices["product"] == "ASH_COATED_OSMIUM"].copy().reset_index(drop=True)
pepperPricesDf = allPrices[allPrices["product"] == "INTARIAN_PEPPER_ROOT"].copy().reset_index(drop=True)

ashTradesDf = allTrades[allTrades["symbol"] == "ASH_COATED_OSMIUM"].copy().reset_index(drop=True)
pepperTradesDf = allTrades[allTrades["symbol"] == "INTARIAN_PEPPER_ROOT"].copy().reset_index(drop=True)

# --------------------------------------------------
# ADD FEATURES
# --------------------------------------------------
ashPricesDf = addBookFeatures(ashPricesDf)
pepperPricesDf = addBookFeatures(pepperPricesDf)

ashTradesDf["globalTs"] = ashTradesDf["day"] * 100000 + ashTradesDf["timestamp"]
pepperTradesDf["globalTs"] = pepperTradesDf["day"] * 100000 + pepperTradesDf["timestamp"]

# --------------------------------------------------
# SAVE CLEAN CSVS
# --------------------------------------------------
allPrices.to_csv(outputDir / "allPrices.csv", index=False)
allTrades.to_csv(outputDir / "allTrades.csv", index=False)

ashPricesDf.to_csv(outputDir / "ashPrices.csv", index=False)
pepperPricesDf.to_csv(outputDir / "pepperPrices.csv", index=False)
ashTradesDf.to_csv(outputDir / "ashTrades.csv", index=False)
pepperTradesDf.to_csv(outputDir / "pepperTrades.csv", index=False)

# --------------------------------------------------
# ANALYSIS-READY FILES
# --------------------------------------------------
ashTradeAggDf = makeTradeAgg(ashTradesDf)
pepperTradeAggDf = makeTradeAgg(pepperTradesDf)

ashAnalysisDf = ashPricesDf.merge(
    ashTradeAggDf,
    on=["day", "timestamp", "globalTs"],
    how="left"
)

pepperAnalysisDf = pepperPricesDf.merge(
    pepperTradeAggDf,
    on=["day", "timestamp", "globalTs"],
    how="left"
)

for df in [ashAnalysisDf, pepperAnalysisDf]:
    for col in ["tradeCount", "totalQty"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

ashAnalysisDf.to_csv(outputDir / "ashAnalysis.csv", index=False)
pepperAnalysisDf.to_csv(outputDir / "pepperAnalysis.csv", index=False)

# --------------------------------------------------
# CHECK
# --------------------------------------------------
print("Saved files to:", outputDir.resolve())

print("\nAll prices days:", sorted(allPrices["day"].dropna().unique()))
print("All trades days:", sorted(allTrades["day"].dropna().unique()))

print("\nashPricesDf unique days:", sorted(ashPricesDf["day"].dropna().unique()))
print("ashTradesDf unique days:", sorted(ashTradesDf["day"].dropna().unique()))

print("\npepperPricesDf unique days:", sorted(pepperPricesDf["day"].dropna().unique()))
print("pepperTradesDf unique days:", sorted(pepperTradesDf["day"].dropna().unique()))

Saved files to: C:\Users\caleb\IMC_Prosperity\ActualComp\cleaned_round1

All prices days: [np.int64(0), np.int64(1), np.int64(2)]
All trades days: [np.int64(0), np.int64(1), np.int64(2)]

ashPricesDf unique days: [np.int64(0), np.int64(1), np.int64(2)]
ashTradesDf unique days: [np.int64(0), np.int64(1), np.int64(2)]

pepperPricesDf unique days: [np.int64(0), np.int64(1), np.int64(2)]
pepperTradesDf unique days: [np.int64(0), np.int64(1), np.int64(2)]


In [17]:
ashPricesDf["day"]

0        0
1        0
2        0
3        0
4        0
        ..
29995   -2
29996   -2
29997   -2
29998   -2
29999   -2
Name: day, Length: 30000, dtype: int64